# TP4: Image Restoration

Gaussian blur, motion blur, Wiener filter deconvolution, Richardson-Lucy iterative restoration, and enhancement pipeline.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import wiener
from scipy.ndimage import convolve
from skimage.restoration import richardson_lucy
from skimage import img_as_float
from skimage.color import rgb2gray
from skimage.util import img_as_ubyte
from skimage.exposure import equalize_hist, rescale_intensity
from skimage.filters import unsharp_mask

%matplotlib inline


In [ ]:
def load_gray(path):
    return cv2.imread(path, cv2.IMREAD_GRAYSCALE)

def show(img, title='', cmap='gray'):
    plt.figure(figsize=(6, 6))
    plt.imshow(img, cmap=cmap)
    plt.title(title)
    plt.axis('off')
    plt.show()


## 1. Load images

In [ ]:
lena  = load_gray("lena.jpg")
noisy = load_gray("noisy.jpg")
blurry = load_gray("02av.jpg")

show(lena,  "Lena")
show(noisy, "Noisy Lena")
show(blurry, "02av (degraded)")


## 2. Gaussian blur then Wiener deconvolution on Lena

In [ ]:
kernel = np.ones((5, 5), np.float32) / 25
lena_blurred = convolve(lena, kernel)
show(lena_blurred, "Lena - Blurred")

lena_restored = wiener(lena_blurred, (5, 5), noise=1)
show(lena_restored, "Lena - Wiener Restored")


## 3. Motion blur then Wiener deconvolution

In [ ]:
motion_kernel = np.zeros((15, 15))
motion_kernel[7, :] = 1 / 15
lena_motion = convolve(lena, motion_kernel)
show(lena_motion, "Lena - Motion Blur")

lena_motion_restored = wiener(lena_motion, (5, 5), noise=1)
show(lena_motion_restored, "Lena - Motion Restored")


## 4. Richardson-Lucy deconvolution on 02av

In [ ]:
image_bgr = cv2.imread("02av.jpg", cv2.IMREAD_COLOR)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
gray_float = img_as_float(rgb2gray(image_rgb))

wiener_filtered = wiener(gray_float, (5, 5))
psf = np.ones((5, 5)) / 25
rl_deconvolved = richardson_lucy(np.clip(wiener_filtered, 0, 1), psf, num_iter=5)
rl_uint8 = img_as_ubyte(np.clip(rl_deconvolved, 0, 1))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image_rgb); axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(wiener_filtered, cmap="gray"); axes[1].set_title("Wiener"); axes[1].axis("off")
axes[2].imshow(rl_uint8, cmap="gray"); axes[2].set_title("Richardson-Lucy"); axes[2].axis("off")
plt.show()


## 5. Restore noisy Lena with Wiener

In [ ]:
noisy_restored = wiener(noisy, (5, 5), noise=1)
show(noisy_restored, "Noisy Lena - Wiener Restored")


## 6. Enhancement pipeline for 02av

Histogram equalization, contrast stretching, unsharp masking, CLAHE, and inpainting.

In [ ]:
equalized = equalize_hist(rl_deconvolved)
rescaled  = rescale_intensity(equalized, in_range='image', out_range=(0, 1))
sharpened = unsharp_mask(rescaled, radius=1.0, amount=1.5)
enhanced  = img_as_ubyte(np.clip(sharpened, 0, 1))

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
clahe_img = clahe.apply(img_as_ubyte(np.clip(rescaled, 0, 1)))
blended = cv2.addWeighted(clahe_img, 0.6, enhanced, 0.4, 0)

defect_mask = cv2.threshold(img_as_ubyte(gray_float), 250, 255, cv2.THRESH_BINARY)[1]
inpainted = cv2.inpaint(blended, defect_mask, 3, cv2.INPAINT_TELEA)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image_rgb); axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(rescaled, cmap="gray"); axes[1].set_title("Contrast Stretched"); axes[1].axis("off")
axes[2].imshow(inpainted, cmap="gray"); axes[2].set_title("Final Enhanced"); axes[2].axis("off")
plt.show()
